# CRNN Data Preparation

This notebook prepares the OCR dataset for CRNN training.

## Objectives

1. Load the OCR dataset
2. Verify dataset integrity
3. Analyze label statistics
4. Split the dataset into training, validation, and testing sets
5. Save the processed dataset


In [1]:
!pip install pandas scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import random
import shutil

import pandas as pd
from sklearn.model_selection import train_test_split

print("Everything installed successfully!")


Everything installed successfully!


In [3]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## Step 1 - Dataset Configuration

In [4]:
# Dataset paths
DATASET_DIR = "OCRDataset"
IMAGE_DIR = os.path.join(DATASET_DIR, "images")
LABEL_FILE = os.path.join(DATASET_DIR, "cleanlabels.csv")

# Split ratios
TRAIN_RATIO = 0.8
VALID_RATIO = 0.1
TEST_RATIO = 0.1

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## Step 2 - Load OCR Dataset

In [5]:
# Load labels
df = pd.read_csv(LABEL_FILE)

print("Dataset loaded successfully!")
print(f"Total samples: {len(df)}")

# Display first 5 rows
df.head()

Dataset loaded successfully!
Total samples: 997


,filename,truth
0,000001.jpg,SAA8967Y
1,000002.jpg,QAB8869B
2,000003.jpg,QM7158B
3,000004.jpg,QKS3847
4,000005.jpg,QTR6769


## Step 3 - Verify Dataset Integrity

In [6]:
# Check dataset information
print("=" * 50)
print("Dataset Information")
print("=" * 50)

print(f"Total samples: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Duplicate filenames
duplicates = df["filename"].duplicated().sum()
print(f"\nDuplicate filenames: {duplicates}")

# Duplicate labels (Allowed)
duplicate_labels = df["truth"].duplicated().sum()
print(f"Duplicate labels: {duplicate_labels}")

# Unique labels
print(f"Unique labels: {df['truth'].nunique()}")

Dataset Information
Total samples: 997
Columns: ['filename', 'truth']

Missing Values:
filename     0
truth       97
dtype: int64

Duplicate filenames: 0
Duplicate labels: 226
Unique labels: 770


## Step 3.1 - Inspect Missing Labels

In [7]:
# Show rows with missing labels

missing_labels = df[df["truth"].isna()]

print(f"Number of missing labels: {len(missing_labels)}")

missing_labels.head(20)

Number of missing labels: 97


,filename,truth
18,000019.jpg,NaN
19,000020.jpg,NaN
22,000023.jpg,NaN
24,000025.jpg,NaN
29,000030.jpg,NaN
32,000033.jpg,NaN
33,000034.jpg,NaN
34,000035.jpg,NaN
35,000036.jpg,NaN
39,000040.jpg,NaN


## Step 3.2 - Remove Unlabeled Samples

In [8]:
# Remove rows with missing labels
df = df.dropna(subset=["truth"]).reset_index(drop=True)

print(f"Remaining samples: {len(df)}")

Remaining samples: 900


In [9]:
print(df.isnull().sum())

filename    0
truth       0
dtype: int64


## Step 4 - Dataset Statistics

In [10]:
# Plate length statistics

df["length"] = df["truth"].str.len()

print("=" * 50)
print("Plate Length Statistics")
print("=" * 50)

print(df["length"].describe())

print("\nPlate Length Distribution:")
print(df["length"].value_counts().sort_index())

Plate Length Statistics
count    900.000000
mean       6.705556
std        0.848326
min        2.000000
25%        7.000000
50%        7.000000
75%        7.000000
max       10.000000
Name: length, dtype: float64

Plate Length Distribution:
length
2       4
3       8
4      19
5      37
6     123
7     673
8      33
9       1
10      2
Name: count, dtype: int64


In [11]:
# Extract all unique characters

characters = sorted(set("".join(df["truth"])))

print("=" * 50)
print("Character Vocabulary")
print("=" * 50)

print(characters)

print(f"\nTotal unique characters: {len(characters)}")

Character Vocabulary
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']

Total unique characters: 35


In [12]:
from collections import Counter

counter = Counter("".join(df["truth"]))

print("=" * 50)
print("Character Frequency")
print("=" * 50)

for char, count in sorted(counter.items()):
    print(f"{char}: {count}")

Character Frequency
0: 246
1: 346
2: 329
3: 379
4: 283
5: 351
6: 362
7: 330
8: 372
9: 361
A: 199
B: 119
C: 113
D: 82
E: 81
F: 48
G: 77
H: 74
I: 14
J: 241
K: 132
L: 74
M: 152
N: 65
O: 13
P: 111
Q: 291
R: 85
S: 113
T: 150
U: 73
V: 140
W: 164
X: 32
Y: 33


## Step 4.1 - Detect Invalid Characters

In [13]:
import re

# Only A-Z and 0-9 are allowed
pattern = r'^[A-Z0-9]+$'

invalid_rows = df[~df["truth"].str.match(pattern, na=False)]

print(f"Number of invalid labels: {len(invalid_rows)}")

invalid_rows[["filename", "truth"]].head(50)

Number of invalid labels: 0


,filename,truth


In [14]:
print("Shortest labels")
display(df[df["length"] <= 4][["filename", "truth"]])

print("\nLongest labels")
display(df[df["length"] >= 9][["filename", "truth"]])

Shortest labels


,filename,truth
14,000015.jpg,QSP2
68,000083.jpg,QP
222,000244.jpg,V105
226,000248.jpg,GT98
229,000251.jpg,BCD
230,000252.jpg,BE57
338,000377.jpg,WAJ4
366,000410.jpg,AAA2
380,000424.jpg,AGN1
385,000429.jpg,JP88



Longest labels


,filename,truth
590,000653.jpg,RIMAU6815
625,000691.jpg,MADANI8481
682,000759.jpg,MADANI8481


## Step 5 - Split Dataset

In [15]:
# ==============================
# Split Dataset (80/10/10)
# ==============================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    shuffle=True
)

print("=" * 50)
print("Dataset Split")
print("=" * 50)

print(f"Training Samples   : {len(train_df)}")
print(f"Validation Samples : {len(valid_df)}")
print(f"Testing Samples    : {len(test_df)}")

Dataset Split
Training Samples   : 720
Validation Samples : 90
Testing Samples    : 90


## Step 6 - Save Train, Validation and Test Sets

In [16]:
# ==============================
# Create Folder Structure
# ==============================

splits = {
    "train": train_df,
    "valid": valid_df,
    "test": test_df
}

for split_name in splits:

    os.makedirs(
        os.path.join(DATASET_DIR, split_name, "images"),
        exist_ok=True
    )

print("Folder structure created successfully!")

Folder structure created successfully!


In [17]:
# ==============================
# Copy Images and Save labels.csv
# ==============================

for split_name, split_df in splits.items():

    # Rename column for CRNN training
    split_df = split_df.rename(columns={"truth": "label"})

    split_folder = os.path.join(DATASET_DIR, split_name)
    image_folder = os.path.join(split_folder, "images")

    # Save labels.csv
    split_df.to_csv(
        os.path.join(split_folder, "labels.csv"),
        index=False
    )

    copied = 0
    missing = 0

    for _, row in split_df.iterrows():

        filename = row["filename"]

        src = os.path.join(IMAGE_DIR, filename)
        dst = os.path.join(image_folder, filename)

        if os.path.exists(src):
            shutil.copy2(src, dst)
            copied += 1
        else:
            missing += 1

    print(f"{split_name.upper()}")
    print(f"  Images Copied : {copied}")
    print(f"  Missing Images: {missing}")
    print("-" * 40)

print("\nDataset split completed successfully!")

TRAIN
  Images Copied : 720
  Missing Images: 0
----------------------------------------
VALID
  Images Copied : 90
  Missing Images: 0
----------------------------------------
TEST
  Images Copied : 90
  Missing Images: 0
----------------------------------------

Dataset split completed successfully!


In [18]:
# ==============================
# Final Verification
# ==============================

for split in ["train", "valid", "test"]:

    image_dir = os.path.join(DATASET_DIR, split, "images")
    label_file = os.path.join(DATASET_DIR, split, "labels.csv")

    image_count = len(os.listdir(image_dir))
    label_count = len(pd.read_csv(label_file))

    print(f"{split.upper()}")
    print(f"Images : {image_count}")
    print(f"Labels : {label_count}")
    print("-" * 30)

TRAIN
Images : 720
Labels : 720
------------------------------
VALID
Images : 90
Labels : 90
------------------------------
TEST
Images : 90
Labels : 90
------------------------------
